# LeetCode 1558 — Minimum Numbers of Function Calls to Make Target Array bằng Python

Link đề: https://leetcode.com/problems/minimum-numbers-of-function-calls-to-make-target-array/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def minOperations(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hoá bài toán "thao tác trên mảng" và giải baseline bằng **BFS trên trạng thái** cho input rất nhỏ.
2. Học kỹ thuật **làm ngược (work backwards)**: từ `nums` quay về mảng toàn 0 bằng các phép ngược "trừ 1" và "chia đôi".
3. Hiểu biểu diễn **nhị phân**: mỗi bit `1` ứng với một lần "+1", mỗi lần "nhân đôi" ứng với một lần dịch trái — từ đó rút ra công thức `Σ popcount(x) + (max bit_length − 1)`.
4. Dùng thành thạo `bin(x).count("1")`, `int.bit_count()`, `int.bit_length()`, phép `& 1`, `>> 1`.
5. Tránh các bẫy: mảng toàn 0, cộng số lần nhân đôi của từng phần tử thay vì lấy max, chia đôi khi còn số lẻ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Có mảng `arr` cùng độ dài, ban đầu toàn `0`. Hàm `modify(arr, op, idx)` có hai kiểu gọi:

- `op == 0`: **tăng 1** một phần tử: `arr[idx] += 1`.
- `op == 1`: **nhân đôi toàn bộ** mảng: `arr[i] *= 2` với mọi `i`.

Trả về **số lần gọi `modify` ít nhất** để biến `arr` thành `nums`. Đáp án đảm bảo vừa số nguyên 32 bit có dấu.

**Ví dụ 1:** `nums = [1,5]` → `5`. Tăng phần tử thứ hai: `[0,1]` (1 lần); nhân đôi 2 lần: `[0,2] → [0,4]` (2 lần); tăng cả hai: `[1,4] → [1,5]` (2 lần). Tổng `1 + 2 + 2 = 5`.

**Ví dụ 2:** `nums = [2,2]` → `3`. Tăng cả hai: `[1,1]` (2 lần); nhân đôi: `[2,2]` (1 lần).

**Ví dụ 3:** `nums = [4,2,5]` → `6`. `[0,0,0] → [1,0,0] → [1,0,1] → [2,0,2] → [2,1,2] → [4,2,4] → [4,2,5]`.

**Ràng buộc:** `1 ≤ nums.length ≤ 10⁵`, `0 ≤ nums[i] ≤ 10⁹`.

In [1]:
import math
import random
import time
from collections import deque
from typing import List

examples = [([1, 5], 5), ([2, 2], 3), ([4, 2, 5], 6)]
for nums, want in examples:
    print(f"nums = {nums} -> mong đợi {want}")

nums = [1, 5] -> mong đợi 5
nums = [2, 2] -> mong đợi 3
nums = [4, 2, 5] -> mong đợi 6


## 3. Kiến thức nền

### 3.1 Biểu diễn nhị phân: `bin`, `bit_length`

`bin(5) = '0b101'`. `x.bit_length()` = số chữ số nhị phân (không tính số 0 đầu), `0.bit_length() = 0`.

In [2]:
for x in [0, 1, 2, 5, 8, 13]:
    print(f"x={x:>2}: bin={bin(x):>7}, bit_length={x.bit_length()}")

x= 0: bin=    0b0, bit_length=0
x= 1: bin=    0b1, bit_length=1
x= 2: bin=   0b10, bit_length=2
x= 5: bin=  0b101, bit_length=3
x= 8: bin= 0b1000, bit_length=4
x=13: bin= 0b1101, bit_length=4


### 3.2 Đếm bit 1 (popcount)

Ba cách: `bin(x).count("1")`, `x.bit_count()` (Python ≥ 3.10), hoặc tự đếm bằng `x & 1` và `x >>= 1`.

In [3]:
def popcount_manual(x):
    c = 0
    while x:
        c += x & 1
        x >>= 1
    return c

for x in [0, 5, 7, 8, 1023]:
    print(x, bin(x).count("1"), x.bit_count(), popcount_manual(x))

0 0 0 0
5 2 2 2
7 3 3 3
8 1 1 1
1023 10 10 10


### 3.3 Nhân đôi = dịch trái, chia đôi = dịch phải

`x * 2 == x << 1`, `x // 2 == x >> 1`. Một số chẵn chia đôi không mất thông tin; số lẻ chia đôi (làm tròn xuống) **mất** bit cuối.

In [4]:
x = 13
print(x << 1, x * 2, "|", x >> 1, x // 2, "| bit cuối của 13:", x & 1)

26 26 | 6 6 | bit cuối của 13: 1


### 3.4 `any` / `all` và `max` trên generator

`any(x % 2 for x in a)` — có phần tử lẻ không; `all(x == 0 for x in a)`; `max(a, default=0)`.

In [5]:
a = [4, 2, 5]
print(any(x % 2 for x in a), all(x == 0 for x in a), max(a))
print(max([], default=0))

True False 5
0


### 3.5 BFS trên trạng thái với `deque` và tuple

Trạng thái là `tuple` (băm được, cho vào `set`). BFS đảm bảo lần đầu chạm đích là số bước ít nhất.

In [6]:
start = (0, 0)
q = deque([(start, 0)])
seen = {start}
s, d = q.popleft()
print("lấy ra:", s, "khoảng cách", d)

lấy ra: (0, 0) khoảng cách 0


## 4. Ý tưởng nền tảng / trực giác

**Nhìn xuôi** thì khó: không biết nên tăng trước hay nhân đôi trước. **Nhìn ngược** (từ `nums` về toàn 0), hai phép ngược là:

- **trừ 1** một phần tử (ngược của tăng 1),
- **chia đôi toàn bộ** — chỉ làm được khi **mọi** phần tử chẵn (ngược của nhân đôi).

Tính tay với `nums = [4, 2, 5]`:

| Bước | Mảng | Hành động ngược | Đếm |
|------|------|-----------------|-----|
| 0 | [4, 2, 5] | có số lẻ (5) ⇒ trừ 1 ở 5 | 1 |
| 1 | [4, 2, 4] | toàn chẵn ⇒ chia đôi | 2 |
| 2 | [2, 1, 2] | có số lẻ (1) ⇒ trừ 1 | 3 |
| 3 | [2, 0, 2] | toàn chẵn ⇒ chia đôi | 4 |
| 4 | [1, 0, 1] | lẻ ⇒ trừ 1 hai lần | 6 |
| 5 | [0, 0, 0] | xong | **6** |

Nhận xét: số lần "trừ 1" = tổng số bit 1 của các phần tử (`4 = 100₂` có 1, `2 = 10₂` có 1, `5 = 101₂` có 2 ⇒ 4); số lần "chia đôi" = `bit_length(5) − 1 = 2`. Tổng `4 + 2 = 6`.

In [7]:
for nums, want in examples:
    pc = sum(bin(x).count("1") for x in nums)
    dbl = max(nums).bit_length() - 1
    print(f"{nums}: tổng bit 1 = {pc}, nhân đôi = {dbl}, tổng = {pc + dbl}, mong đợi {want}")

[1, 5]: tổng bit 1 = 3, nhân đôi = 2, tổng = 5, mong đợi 5
[2, 2]: tổng bit 1 = 2, nhân đôi = 1, tổng = 3, mong đợi 3
[4, 2, 5]: tổng bit 1 = 4, nhân đôi = 2, tổng = 6, mong đợi 6


## 5. Cách 1 — Brute Force / baseline: BFS trên trạng thái

**Ý tưởng.** Trạng thái là mảng hiện tại. Từ trạng thái `s`, có `n` cách "tăng 1" và 1 cách "nhân đôi". Ta chỉ giữ các trạng thái **không vượt** `nums` ở mọi vị trí (vượt rồi thì không quay lại được vì không có phép giảm). BFS từ `(0,...,0)` tới `nums`.

Số trạng thái `≤ Π(nums[i] + 1)` — chỉ dùng được với mảng rất nhỏ (ví dụ `n ≤ 3`, giá trị `≤ 10`). Đây là **chuẩn đối chiếu đáng tin** vì nó thật sự thử mọi chuỗi thao tác.

In [8]:
def brute_bfs(nums):
    target = tuple(nums)
    start = tuple(0 for _ in nums)
    if start == target:
        return 0
    q = deque([start])
    dist = {start: 0}
    while q:
        s = q.popleft()
        d = dist[s]
        nxt = []
        for i in range(len(s)):                          # op 0: tăng 1 tại i
            if s[i] < target[i]:
                nxt.append(s[:i] + (s[i] + 1,) + s[i + 1:])
        dbl = tuple(2 * x for x in s)                    # op 1: nhân đôi tất cả
        if dbl != s and all(a <= b for a, b in zip(dbl, target)):
            nxt.append(dbl)
        for t in nxt:
            if t not in dist:
                dist[t] = d + 1
                if t == target:
                    return d + 1
                q.append(t)
    return -1

for nums, want in examples:
    print(nums, brute_bfs(nums), "mong đợi", want)

[1, 5] 5 mong đợi 5
[2, 2] 3 mong đợi 3
[4, 2, 5] 6 mong đợi 6


In [9]:
# Nút thắt: số trạng thái bùng nổ theo tích các giá trị
for nums in [[7, 7], [15, 15], [20, 20, 20], [40, 40, 40], [30, 30, 30, 30]]:
    t0 = time.perf_counter()
    r = brute_bfs(nums)
    print(f"{nums}: {r} ({time.perf_counter() - t0:.3f}s), số trạng thái tối đa {math.prod(x + 1 for x in nums)}")

[7, 7]: 8 (0.000s), số trạng thái tối đa 64
[15, 15]: 11 (0.001s), số trạng thái tối đa 256
[20, 20, 20]: 10 (0.012s), số trạng thái tối đa 9261
[40, 40, 40]: 11 (0.041s), số trạng thái tối đa 68921


[30, 30, 30, 30]: 20 (2.600s), số trạng thái tối đa 923521


**Độ phức tạp:** `O(n · Π(nums[i]+1))` — hoàn toàn bất khả thi với `nums[i]` tới `10⁹`.

**Nút thắt:** ta khám phá **mọi** thứ tự thao tác, trong khi thứ tự tối ưu có cấu trúc rất rõ (nhìn ngược). Cần một chiến lược tham lam / công thức.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — làm ngược, tham lam.** Khi đi ngược:
- *Nếu* còn phần tử **lẻ** *thì* bắt buộc phải "trừ 1" ở nó (chia đôi chỉ làm được khi tất cả chẵn). Trừ 1 ở phần tử lẻ là không tránh được.
- *Nếu* **mọi** phần tử chẵn (và chưa toàn 0) *thì* "chia đôi" ngay là tốt nhất: một lần chia đôi xử lý cả mảng, còn trừ 1 từng chút ở số chẵn chỉ tốn thêm.

**Insight 2 — đếm theo bit.** Nhìn mỗi phần tử ở dạng nhị phân. Mỗi lần "chia đôi" dịch phải mọi số; mỗi lần "trừ 1" xoá bit cuối bằng 1 của một số. Do đó:
- Tổng số lần "trừ 1" = tổng số bit 1 = `Σ popcount(nums[i])`.
- Số lần "chia đôi" = số lần cần dịch phải để số **lớn nhất** về 0 sau khi đã xoá bit, tức `bit_length(max) − 1` (bit cao nhất xoá bằng một lần "trừ 1" ở cuối).

**Công thức:** `answer = Σ popcount(x) + max(0, max(nums).bit_length() − 1)`.

**Invariant khi mô phỏng ngược:** sau mỗi vòng "xoá bit lẻ + chia đôi", số đã đếm cộng với `Σ popcount + (max bit_length − 1)` của mảng hiện tại luôn bằng đáp án.

In [10]:
nums = [4, 2, 5]
for x in nums:
    print(f"{x} = {bin(x)[2:]:>3}₂: popcount = {x.bit_count()}, bit_length - 1 = {x.bit_length() - 1}")
print("đáp án =", sum(x.bit_count() for x in nums), "+", max(nums).bit_length() - 1)

4 = 100₂: popcount = 1, bit_length - 1 = 2
2 =  10₂: popcount = 1, bit_length - 1 = 1
5 = 101₂: popcount = 2, bit_length - 1 = 2
đáp án = 4 + 2


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng ngược

Từng đoạn:
1. `a = list(nums)`, `ops = 0`.
2. Lặp: trừ 1 mọi phần tử lẻ (mỗi cái +1 vào `ops`).
3. Nếu mảng toàn 0 ⇒ dừng. Ngược lại chia đôi mọi phần tử, `ops += 1`.

Mỗi vòng giá trị lớn nhất giảm một nửa ⇒ `O(log max)` vòng, mỗi vòng `O(n)`.

In [11]:
def reverse_simulation(nums):
    a = list(nums)
    ops = 0
    while True:
        for i in range(len(a)):
            if a[i] & 1:
                a[i] -= 1
                ops += 1
        if not any(a):
            return ops
        for i in range(len(a)):
            a[i] >>= 1
        ops += 1

for nums, want in examples:
    print(nums, reverse_simulation(nums), "mong đợi", want)

[1, 5] 5 mong đợi 5
[2, 2] 3 mong đợi 3
[4, 2, 5] 6 mong đợi 6


Code nộp LeetCode cho cách 2:

In [12]:
class SolutionSimulate:
    def minOperations(self, nums: List[int]) -> int:
        a = list(nums)
        ops = 0
        while True:
            for i, x in enumerate(a):
                if x & 1:
                    a[i] = x - 1
                    ops += 1
            if not any(a):
                return ops
            a = [x >> 1 for x in a]
            ops += 1

print([SolutionSimulate().minOperations(x) for x, _ in examples])

[5, 3, 6]


**Độ phức tạp:** Time `O(n · log M)` với `M = max(nums) ≤ 10⁹` (≤ 30 vòng), Space `O(n)` cho bản sao.

### 7.2 Cách 3 — Đếm bit thủ công cho từng phần tử

Không mô phỏng cả mảng; với mỗi `x`, đếm số bit 1 (`adds`) và số lần dịch phải (`shifts = bit_length − 1`). Đáp án = `Σ adds + max shifts`.

In [13]:
def per_element(nums):
    total_adds = 0
    max_shifts = 0
    for x in nums:
        shifts = 0
        while x:
            total_adds += x & 1
            x >>= 1
            if x:
                shifts += 1
        max_shifts = max(max_shifts, shifts)
    return total_adds + max_shifts

print([per_element(x) for x, _ in examples])

[5, 3, 6]


Code nộp LeetCode cho cách 3:

In [14]:
class SolutionPerElement:
    def minOperations(self, nums: List[int]) -> int:
        adds, shifts = 0, 0
        for x in nums:
            s = 0
            while x:
                adds += x & 1
                x >>= 1
                if x:
                    s += 1
            shifts = max(shifts, s)
        return adds + shifts

print([SolutionPerElement().minOperations(x) for x, _ in examples])

[5, 3, 6]


**Độ phức tạp:** Time `O(n · log M)`, Space `O(1)`.

### 7.3 Cách 4 — Công thức với hàm dựng sẵn

`sum(x.bit_count() for x in nums) + max(0, max(nums).bit_length() − 1)`. Dùng `max(0, ...)` để xử lý mảng toàn 0 (khi đó `bit_length = 0`). Với Python cũ hơn 3.10 thay `x.bit_count()` bằng `bin(x).count("1")`.

In [15]:
def formula(nums):
    return sum(x.bit_count() for x in nums) + max(0, max(nums).bit_length() - 1)

print([formula(x) for x, _ in examples], formula([0, 0, 0]))

[5, 3, 6] 0


Code nộp LeetCode cho cách 4:

In [16]:
class SolutionFormula:
    def minOperations(self, nums: List[int]) -> int:
        return sum(bin(x).count("1") for x in nums) + max(0, max(nums).bit_length() - 1)

print([SolutionFormula().minOperations(x) for x, _ in examples])

[5, 3, 6]


**Độ phức tạp:** Time `O(n · log M)` (đếm bit; thực tế rất nhanh vì chạy trong C), Space `O(1)` (bản `bin` tạo chuỗi tạm `O(log M)`).

## 8. TRACE từng bước

Trace mô phỏng ngược với `nums = [1, 5]`:

| Vòng | Trước | Trừ 1 ở số lẻ | Sau trừ | Chia đôi? | ops |
|------|-------|---------------|---------|-----------|-----|
| 1 | [1, 5] | 1, 5 (2 lần) | [0, 4] | có ⇒ [0, 2] | 3 |
| 2 | [0, 2] | — | [0, 2] | có ⇒ [0, 1] | 4 |
| 3 | [0, 1] | 1 (1 lần) | [0, 0] | dừng | **5** |

In [17]:
def trace_reverse(nums):
    a = list(nums)
    ops = 0
    rnd = 0
    print(f"bắt đầu: {a}")
    while True:
        rnd += 1
        odd = [i for i, x in enumerate(a) if x & 1]
        for i in odd:
            a[i] -= 1
        ops += len(odd)
        print(f"  vòng {rnd}: trừ 1 tại chỉ số {odd} -> {a}, ops = {ops}")
        if not any(a):
            print(f"  toàn 0 -> dừng, đáp án {ops}")
            return ops
        a = [x >> 1 for x in a]
        ops += 1
        print(f"           chia đôi -> {a}, ops = {ops}")

trace_reverse([1, 5])

bắt đầu: [1, 5]
  vòng 1: trừ 1 tại chỉ số [0, 1] -> [0, 4], ops = 2
           chia đôi -> [0, 2], ops = 3
  vòng 2: trừ 1 tại chỉ số [] -> [0, 2], ops = 3
           chia đôi -> [0, 1], ops = 4
  vòng 3: trừ 1 tại chỉ số [1] -> [0, 0], ops = 5
  toàn 0 -> dừng, đáp án 5


5

In [18]:
trace_reverse([4, 2, 5])

bắt đầu: [4, 2, 5]
  vòng 1: trừ 1 tại chỉ số [2] -> [4, 2, 4], ops = 1
           chia đôi -> [2, 1, 2], ops = 2
  vòng 2: trừ 1 tại chỉ số [1] -> [2, 0, 2], ops = 3
           chia đôi -> [1, 0, 1], ops = 4
  vòng 3: trừ 1 tại chỉ số [0, 2] -> [0, 0, 0], ops = 6
  toàn 0 -> dừng, đáp án 6


6

Trace cách đếm bit (cách 3) cho từng phần tử:

In [19]:
def trace_bits(nums):
    adds, shifts = 0, 0
    for x in nums:
        a, s = x.bit_count(), max(0, x.bit_length() - 1)
        adds += a
        shifts = max(shifts, s)
        print(f"x={x:>3} ({bin(x)[2:]:>8}₂): +{a} lần tăng, cần {s} lần nhân đôi | Σadds={adds}, max shifts={shifts}")
    print("đáp án =", adds + shifts)
    return adds + shifts

trace_bits([4, 2, 5])
trace_bits([3, 2, 2, 4])

x=  4 (     100₂): +1 lần tăng, cần 2 lần nhân đôi | Σadds=1, max shifts=2
x=  2 (      10₂): +1 lần tăng, cần 1 lần nhân đôi | Σadds=2, max shifts=2
x=  5 (     101₂): +2 lần tăng, cần 2 lần nhân đôi | Σadds=4, max shifts=2
đáp án = 6
x=  3 (      11₂): +2 lần tăng, cần 1 lần nhân đôi | Σadds=2, max shifts=1
x=  2 (      10₂): +1 lần tăng, cần 1 lần nhân đôi | Σadds=3, max shifts=1
x=  2 (      10₂): +1 lần tăng, cần 1 lần nhân đôi | Σadds=4, max shifts=1
x=  4 (     100₂): +1 lần tăng, cần 2 lần nhân đôi | Σadds=5, max shifts=2
đáp án = 7


7

## 9. Code nộp LeetCode

In [20]:
class Solution:
    def minOperations(self, nums: List[int]) -> int:
        # số lần +1 = tổng số bit 1; số lần nhân đôi = bit_length(max) - 1 (nếu max > 0)
        adds = sum(bin(x).count("1") for x in nums)
        doubles = max(0, max(nums).bit_length() - 1)
        return adds + doubles

for nums, want in examples:
    got = Solution().minOperations(nums)
    print(got, want)
    assert got == want

5 5
3 3
6 6


## 10. Vì sao thuật toán đúng

Đặt **thế năng** `Φ(arr) = Σ popcount(arr[i]) + max(0, bit_length(max(arr)) − 1)`. Ta có `Φ([0,…,0]) = 0` và `Φ(nums)` chính là công thức.

**Cận dưới — mỗi lời gọi làm `Φ` tăng tối đa 1.**
- *Nhân đôi cả mảng:* popcount mỗi phần tử không đổi (chỉ dịch bit); `bit_length(max)` tăng đúng 1 (nếu `max > 0`) ⇒ `ΔΦ = 1` (hoặc `0` nếu mảng toàn 0).
- *Tăng 1 phần tử `a → a + 1`:* `popcount(a + 1) ≤ popcount(a) + 1`. Nếu `bit_length(max)` không đổi thì `ΔΦ ≤ 1`. Nếu nó tăng thì `a + 1 = 2^j` là max mới với `j ≥ 1`, tức `a = 2^j − 1` có `j` bit 1 còn `a + 1` chỉ có 1 bit 1 ⇒ phần popcount giảm `j − 1 ≥ 0`, phần bit_length tăng 1 ⇒ `ΔΦ ≤ 1`. (Trường hợp `0 → 1` từ mảng toàn 0: popcount +1, phần bit_length vẫn 0 ⇒ `ΔΦ = 1`.)

Từ `Φ = 0` lên `Φ(nums)` mà mỗi bước tăng ≤ 1 ⇒ cần **ít nhất** `Φ(nums)` lời gọi.

**Cận trên — có kế hoạch đúng `Φ(nums)` lời gọi.** Gọi `D = bit_length(max) − 1`. Duyệt bit `b = D, D−1, …, 0`: tăng 1 cho mọi phần tử có bit `b` bằng 1, rồi nhân đôi (trừ khi `b = 0`). Số lời gọi `= Σ popcount + D` và mảng thu được đúng là `nums` (giống cách đọc số nhị phân từ bit cao: `x = (…((b_D)·2 + b_{D−1})·2 + …) + b_0`). Bài tập 4 cài đặt và kiểm tra kế hoạch này.

**Tham lam ngược** chính là kế hoạch trên đi theo chiều ngược lại: số lẻ (bit cuối = 1) bắt buộc "trừ 1"; toàn chẵn thì "chia đôi". Kiểm chứng thêm bằng BFS vét cạn dưới đây.

In [21]:
# Kiểm chứng vét cạn: mọi mảng độ dài 1..2 với giá trị 0..12
from itertools import product
cnt = 0
for n in (1, 2):
    for nums in product(range(13), repeat=n):
        assert formula(list(nums)) == brute_bfs(list(nums)), nums
        cnt += 1
print(f"công thức khớp BFS trên {cnt} mảng")

công thức khớp BFS trên 182 mảng


## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `M = max(nums) ≤ 10⁹` (nên `log₂ M ≤ 30`).

| Cách | Time | Space |
|------|------|-------|
| BFS trạng thái | `O(n · Π(nums[i]+1))` | như trên |
| Mô phỏng ngược | `O(n log M)` | `O(n)` |
| Đếm bit từng phần tử | `O(n log M)` | `O(1)` |
| Công thức dựng sẵn | `O(n log M)` (hằng số nhỏ) | `O(1)` |

Đáp án lớn nhất: `30 · 10⁵ + 29` ≈ `3·10⁶` — vừa `int` 32 bit như đề nói.

In [22]:
rng = random.Random(1558)
big = [rng.randint(0, 10**9) for _ in range(10**5)]
for name, f in [("simulate", SolutionSimulate()), ("per_element", SolutionPerElement()), ("formula", SolutionFormula()), ("Solution", Solution())]:
    t0 = time.perf_counter()
    r = f.minOperations(big)
    print(f"{name:12s}: {r} ({time.perf_counter() - t0:.3f}s)")

simulate    : 1483856 (0.282s)


per_element : 1483856 (0.229s)
formula     : 1483856 (0.031s)
Solution    : 1483856 (0.030s)


## 12. Bộ test

In [23]:
METHODS = {
    "reverse_simulation": reverse_simulation,
    "per_element": per_element,
    "formula": formula,
    "SolutionSimulate": lambda a: SolutionSimulate().minOperations(a),
    "SolutionPerElement": lambda a: SolutionPerElement().minOperations(a),
    "SolutionFormula": lambda a: SolutionFormula().minOperations(a),
    "Solution": lambda a: Solution().minOperations(a),
}
for nums, want in examples:
    assert brute_bfs(nums) == want
    for name, f in METHODS.items():
        assert f(nums) == want, (name, nums)
print("ví dụ đề: OK")

ví dụ đề: OK


In [24]:
# edge case
edge = [
    ([0], 0),                     # đã đúng
    ([0, 0, 0], 0),               # toàn 0 (bẫy bit_length - 1 = -1)
    ([1], 1),
    ([8], 4),                     # +1 rồi nhân đôi 3 lần
    ([7], 5),                     # 111₂: 3 lần +1, 2 lần nhân đôi
    ([0, 1, 0], 1),
    ([10**9], (10**9).bit_count() + (10**9).bit_length() - 1),
    ([3, 2, 2, 4], 7),
]
for nums, want in edge:
    for name, f in METHODS.items():
        assert f(nums) == want, (name, nums, f(nums))
print("edge case: OK")

edge case: OK


In [25]:
# ngẫu nhiên có seed: đối chiếu với BFS vét cạn trên mảng nhỏ
rng = random.Random(2021)
for _ in range(300):
    n = rng.randint(1, 3)
    nums = [rng.randint(0, 9) for _ in range(n)]
    ref = brute_bfs(nums)
    for name, f in METHODS.items():
        assert f(nums) == ref, (name, nums)
print("300 mảng nhỏ ngẫu nhiên (seed 2021): khớp BFS")

300 mảng nhỏ ngẫu nhiên (seed 2021): khớp BFS


In [26]:
# ngẫu nhiên cỡ lớn trong ràng buộc: các cách nhanh khớp nhau
rng = random.Random(7)
for _ in range(200):
    nums = [rng.choice([0, rng.randint(0, 10**9), rng.randint(0, 64)]) for _ in range(rng.randint(1, 60))]
    ref = reverse_simulation(nums)
    for name, f in METHODS.items():
        assert f(nums) == ref, (name, nums)
print("200 mảng ngẫu nhiên lớn: OK")

200 mảng ngẫu nhiên lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trường hợp mảng toàn 0

`max(nums).bit_length() − 1 = −1` khi mọi phần tử bằng 0 ⇒ trả `−1`.

In [27]:
def wrong_all_zero(nums):
    return sum(x.bit_count() for x in nums) + max(nums).bit_length() - 1   # SAI khi max = 0

for nums in [[0], [0, 0, 0], [4, 2, 5]]:
    print(nums, "| sai:", wrong_all_zero(nums), "| đúng:", Solution().minOperations(nums))

[0] | sai: -1 | đúng: 0
[0, 0, 0] | sai: -1 | đúng: 0
[4, 2, 5] | sai: 6 | đúng: 6


Sửa: `max(0, max(nums).bit_length() − 1)`.

### Lỗi 2 — Cộng số lần nhân đôi của **từng** phần tử

Nhân đôi tác động lên **cả mảng** cùng lúc, nên số lần nhân đôi là **max**, không phải tổng.

In [28]:
def wrong_sum_doubles(nums):
    return sum(x.bit_count() + max(0, x.bit_length() - 1) for x in nums)   # SAI: cộng nhân đôi từng phần tử

for nums, want in examples:
    print(nums, "| sai:", wrong_sum_doubles(nums), "| đúng:", want)

[1, 5] | sai: 5 | đúng: 5
[2, 2] | sai: 4 | đúng: 3
[4, 2, 5] | sai: 9 | đúng: 6


Sửa: tách riêng `Σ popcount` và `max(bit_length − 1)`.

### Lỗi 3 — Mô phỏng ngược nhưng chia đôi khi vẫn còn số lẻ

Chia đôi bằng `>> 1` khi có số lẻ sẽ **âm thầm làm mất** bit 1 cuối ⇒ đếm thiếu.

In [29]:
def wrong_halve_odd(nums):
    a = list(nums)
    ops = 0
    while any(a):
        if all(x % 2 == 0 for x in a):
            a = [x >> 1 for x in a]
            ops += 1
        else:
            a = [x >> 1 for x in a]          # SAI: chia đôi luôn, bỏ qua bước trừ 1 ở số lẻ
            ops += 1
    return ops

for nums, want in examples:
    print(nums, "| sai:", wrong_halve_odd(nums), "| đúng:", want)

[1, 5] | sai: 3 | đúng: 5
[2, 2] | sai: 2 | đúng: 3
[4, 2, 5] | sai: 3 | đúng: 6


Sửa: khi có số lẻ, **trừ 1** ở từng số lẻ (đếm mỗi lần), rồi mới chia đôi.

### Lỗi 4 — Dùng `bit_length()` thay vì `bit_length() − 1`

Số lần nhân đôi để có bit cao nhất ở vị trí `L − 1` là `L − 1` (bit đầu tiên tạo bằng "+1", không cần nhân đôi trước nó).

In [30]:
def wrong_off_by_one(nums):
    return sum(x.bit_count() for x in nums) + max(nums).bit_length()   # SAI: thừa 1

for nums, want in examples + [([1], 1)]:
    print(nums, "| sai:", wrong_off_by_one(nums), "| đúng:", want)

[1, 5] | sai: 6 | đúng: 5
[2, 2] | sai: 4 | đúng: 3
[4, 2, 5] | sai: 7 | đúng: 6
[1] | sai: 2 | đúng: 1


Sửa: `bit_length() − 1` (và kẹp về 0).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thao tác **"+1 một phần tử"** và **"×2 cả mảng"** ⇒ nghĩ tới **nhị phân**: +1 tạo bit, ×2 dịch trái.
- Bài "tối thiểu số thao tác để từ 0 tới X" với các phép tiến ⇒ thử **làm ngược** từ X: phép ngược thường tham lam được (giống Broken Calculator LC 991, Reduce a number to zero LC 1342).
- Có phép áp dụng **đồng thời lên tất cả** ⇒ chi phí của nó tính theo **max**, phép riêng lẻ tính theo **tổng**.
- Luôn thử mảng toàn 0 / một phần tử khi có `max(...) − 1`.

## 15. Biến thể của bài

1. **Một phần tử** (LC 1342 *Number of Steps to Reduce a Number to Zero*): đáp án `popcount(x) + bit_length(x) − 1`.
2. **Nhân `k` thay vì nhân 2** (bài tập 5): tổng chữ số cơ số `k` + (số chữ số lớn nhất − 1).
3. **Broken Calculator (LC 991)**: từ `X` tới `Y` bằng ×2 và −1 — làm ngược từ `Y` (chia đôi khi chẵn, +1 khi lẻ).
4. **Mỗi phép +1 có thể cộng vào một đoạn liên tiếp** — trở thành bài khác (LC 1526 *Minimum Number of Increments on Subarrays*).
5. **Chi phí khác nhau** cho hai phép: vẫn làm ngược, nhưng phải cân nhắc trừ nhiều lần thay vì chia đôi.

In [31]:
# Biến thể 1 (LC 1342): số bước đưa x về 0 bằng "chia 2 nếu chẵn, trừ 1 nếu lẻ"
def steps_to_zero(x):
    return x.bit_count() + max(0, x.bit_length() - 1)

print([steps_to_zero(x) for x in [14, 8, 123, 0]], "(LC 1342: 14 -> 6, 8 -> 4, 123 -> 12)")

[6, 4, 12, 0] (LC 1342: 14 -> 6, 8 -> 4, 123 -> 12)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. BFS trạng thái | thử mọi chuỗi thao tác | mũ | mũ | kiểm chứng, giá trị rất nhỏ |
| 2. Mô phỏng ngược | trừ 1 số lẻ, chia đôi khi toàn chẵn | `O(n log M)` | `O(n)` | dễ nghĩ ra từ gợi ý đề |
| 3. Đếm bit thủ công | adds = bit 1, shifts = max | `O(n log M)` | `O(1)` | phỏng vấn cần tự cài |
| 4. Công thức dựng sẵn | `Σ popcount + max bit_length − 1` | `O(n log M)` | `O(1)` | **bản nộp** |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `minOperations([3])`, `minOperations([8])`, `minOperations([0, 0])` lần lượt bằng bao nhiêu?

**Bài 2 (tính tay):** Chạy tay mô phỏng ngược trên `nums = [7, 3]`: ghi mảng sau mỗi bước và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `ex3_popcount(x)` đếm số bit 1 bằng `& 1` và `>> 1` (không dùng `bin`/`bit_count`).

**Bài 4 (tự viết lại):** Viết `ex4_plan(nums)` trả về **danh sách thao tác xuôi** tối ưu, mỗi thao tác là `("inc", i)` hoặc `("double",)`. Áp dụng lên mảng 0 phải ra `nums` và độ dài phải bằng đáp án.

**Bài 5 (mini challenge):** Viết `ex5_min_ops_k(nums, k)` cho phiên bản phép thứ hai là **nhân `k` toàn bộ mảng** (`k ≥ 2`).

In [32]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_popcount(x):
    c = 0
    while x:
        # TODO: cộng bit cuối của x vào c
        # TODO: dịch phải x một bit
        pass
    return c

In [33]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_plan(nums):
    # TODO
    raise NotImplementedError

def ex5_min_ops_k(nums, k):
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `3 = 11₂`, `8 = 1000₂`. (2) `3`: 2 bit 1 + 1 nhân đôi; `8`: 1 + 3. (3) `3`, `4`, `0`.

**Bài 2.** (1) `[7,3]` toàn lẻ ⇒ trừ 2 lần → `[6,2]`, chia đôi → `[3,1]`. (2) Trừ 2 lần → `[2,0]`, chia đôi → `[1,0]`. (3) Trừ 1 → `[0,0]`. Tổng `2+1+2+1+1 = 7` (= `3 + 2` bit 1 + `2` nhân đôi).

**Bài 3.** (1) `c += x & 1`. (2) `x >>= 1`. (3) Vòng dừng khi `x == 0`.

**Bài 4.** (1) `D = bit_length(max) − 1`. (2) Với bit `b` từ `D` xuống `0`: tăng mọi `i` có bit `b` của `nums[i]` bằng 1; nếu `b > 0` thì nhân đôi. (3) Mảng toàn 0 ⇒ kế hoạch rỗng.

**Bài 5.** (1) Làm ngược: phần dư `x mod k` phải trừ từng đơn vị. (2) Số lần "+1" = tổng chữ số của `x` trong cơ số `k`. (3) Số lần nhân `k` = (số chữ số cơ số `k` của `max`) − 1. Kiểm chứng bằng BFS sửa `2` thành `k`.

## 19. Đáp án tham khảo cho bài tập

In [34]:
# Bài 1
a1 = (Solution().minOperations([3]), Solution().minOperations([8]), Solution().minOperations([0, 0]))
print("Bài 1:", a1)
assert a1 == (3, 4, 0)

# Bài 2
a2 = trace_reverse([7, 3])
assert a2 == 7 == brute_bfs([7, 3])

Bài 1: (3, 4, 0)
bắt đầu: [7, 3]
  vòng 1: trừ 1 tại chỉ số [0, 1] -> [6, 2], ops = 2
           chia đôi -> [3, 1], ops = 3
  vòng 2: trừ 1 tại chỉ số [0, 1] -> [2, 0], ops = 5
           chia đôi -> [1, 0], ops = 6
  vòng 3: trừ 1 tại chỉ số [0] -> [0, 0], ops = 7
  toàn 0 -> dừng, đáp án 7


In [35]:
# Bài 3
def ex3_popcount_sol(x):
    c = 0
    while x:
        c += x & 1
        x >>= 1
    return c

assert all(ex3_popcount_sol(x) == bin(x).count("1") for x in range(5000))
assert ex3_popcount_sol(10**9) == (10**9).bit_count()
print("Bài 3: OK")

Bài 3: OK


In [36]:
# Bài 4
def ex4_plan_sol(nums):
    M = max(nums)
    if M == 0:
        return []
    D = M.bit_length() - 1
    plan = []
    for b in range(D, -1, -1):
        for i, x in enumerate(nums):
            if (x >> b) & 1:
                plan.append(("inc", i))
        if b > 0:
            plan.append(("double",))
    return plan

def apply_plan(n, plan):
    arr = [0] * n
    for op in plan:
        if op[0] == "inc":
            arr[op[1]] += 1
        else:
            arr = [2 * x for x in arr]
    return arr

rng4 = random.Random(4)
for _ in range(500):
    nums = [rng4.randint(0, 300) for _ in range(rng4.randint(1, 6))]
    plan = ex4_plan_sol(nums)
    assert apply_plan(len(nums), plan) == nums
    assert len(plan) == Solution().minOperations(nums)
print("Bài 4: OK, kế hoạch cho [1,5]:", ex4_plan_sol([1, 5]))

Bài 4: OK, kế hoạch cho [1,5]: [('inc', 1), ('double',), ('double',), ('inc', 0), ('inc', 1)]


In [37]:
# Bài 5
def ex5_min_ops_k_sol(nums, k):
    adds, max_digits = 0, 0
    for x in nums:
        d = 0
        while x:
            adds += x % k
            x //= k
            d += 1
        max_digits = max(max_digits, d)
    return adds + max(0, max_digits - 1)

def brute_bfs_k(nums, k):
    target = tuple(nums)
    start = tuple(0 for _ in nums)
    if start == target:
        return 0
    q = deque([start])
    dist = {start: 0}
    while q:
        s = q.popleft()
        nxt = [s[:i] + (s[i] + 1,) + s[i + 1:] for i in range(len(s)) if s[i] < target[i]]
        m = tuple(k * x for x in s)
        if m != s and all(a <= b for a, b in zip(m, target)):
            nxt.append(m)
        for t in nxt:
            if t not in dist:
                dist[t] = dist[s] + 1
                if t == target:
                    return dist[t]
                q.append(t)

assert all(ex5_min_ops_k_sol(nums, 2) == Solution().minOperations(nums) for nums, _ in examples)
rng5 = random.Random(5)
for _ in range(300):
    k = rng5.randint(2, 5)
    nums = [rng5.randint(0, 20) for _ in range(rng5.randint(1, 2))]
    assert ex5_min_ops_k_sol(nums, k) == brute_bfs_k(nums, k), (nums, k)
print("Bài 5: OK, ví dụ k=3:", ex5_min_ops_k_sol([4, 2, 5], 3), "| k=10:", ex5_min_ops_k_sol([123], 10))

Bài 5: OK, ví dụ k=3: 8 | k=10: 8


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def minOperations(self, nums):
        adds = sum(bin(x).count("1") for x in nums)          # mỗi bit 1 = một lần +1
        doubles = max(0, max(nums).bit_length() - 1)         # nhân đôi chung cho cả mảng
        return adds + doubles
```

**Template "làm ngược + nhị phân":** phép riêng lẻ ⇒ cộng theo từng phần tử; phép đồng loạt ⇒ lấy max. Ngược lại: số lẻ phải trừ 1, toàn chẵn thì chia đôi.

**Checklist trước khi submit:**
- [ ] Số lần nhân đôi lấy **max**, không cộng.
- [ ] `bit_length() − 1` và kẹp `max(0, …)` cho mảng toàn 0.
- [ ] Nếu mô phỏng ngược: chỉ chia đôi khi **mọi** phần tử chẵn.
- [ ] Thử nhanh: `[1,5] → 5`, `[2,2] → 3`, `[4,2,5] → 6`, `[0] → 0`, `[8] → 4`.